In [ ]:
%%sql -r dataframe_2
use FOOD_DELIVERY_DB;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE STAGE LANDING_STAGE
URL = 'azure://fdsmanasvi.blob.core.windows.net/landing'
CREDENTIALS = (
AZURE_SAS_TOKEN =' sp=rl&st=2026-05-29T06:21:50Z&se=2026-06-01T14:36:50Z&spr=https&sv=2026-02-06&sr=c&sig=eSKvdb0uXOZ9nHnkwVMfqFTVm5VCDmFJbQ2lvigC4VU%3D'
);

In [ ]:
%%sql -r dataframe_3
LIST @LANDING_STAGE;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE STREAM STREAM_ORDER_DETAILS
ON TABLE BRONZE.ORDER_DETAILS;

In [ ]:
%%sql -r dataframe_5
SELECT COUNT(*)
FROM STREAM_ORDER_DETAILS;

In [ ]:
%%sql -r dataframe_6
COPY INTO BRONZE.ORDER_DETAILS
(
ORDER_ID,
CUSTOMER_ID,
RESTAURANT_ID,
AGENT_ID,
ORDER_PLACED_AT,
ORDER_ACCEPTED_AT,
ORDER_DELIVERED_AT,
ORDER_STATUS,
TOTAL_AMOUNT,
DISCOUNT_AMOUNT,
DELIVERY_FEE,
TAX_AMOUNT,
FINAL_AMOUNT,
DELIVERY_DISTANCE_KM,
ESTIMATED_DELIVERY_TIME,
ACTUAL_DELIVERY_TIME,
DELIVERY_CITY,
DELIVERY_PINCODE,
ORDER_SOURCE,
PROMO_CODE,
INGESTION_TS,
SOURCE_FILE_NAME
)
FROM
(
SELECT
$1,$2,$3,$4,$5,$6,$7,$8,$9,$10,
$11,$12,$13,$14,$15,$16,$17,$18,$19,$20,
CURRENT_TIMESTAMP(),
METADATA$FILENAME
FROM @LANDING_STAGE/Batch_3_Order_Details.csv
)
FILE_FORMAT = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_8
SELECT *
FROM STREAM_ORDER_DETAILS
LIMIT 10;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE PROCEDURE PROCESS_ORDER_STREAM()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    ------------------------------------------------------------------
    -- LOAD STREAM DATA INTO TEMP TABLE
    ------------------------------------------------------------------

    CREATE OR REPLACE TEMP TABLE TMP_ORDER_STREAM AS
    SELECT
        ORDER_ID,
        CUSTOMER_ID,
        RESTAURANT_ID,
        AGENT_ID,
        ORDER_PLACED_AT,
        ORDER_ACCEPTED_AT,
        ORDER_DELIVERED_AT,
        ORDER_STATUS,
        TOTAL_AMOUNT,
        DISCOUNT_AMOUNT,
        DELIVERY_FEE,
        TAX_AMOUNT,
        FINAL_AMOUNT,
        DELIVERY_DISTANCE_KM,
        ESTIMATED_DELIVERY_TIME,
        ACTUAL_DELIVERY_TIME,
        DELIVERY_CITY,
        DELIVERY_PINCODE,
        ORDER_SOURCE,
        PROMO_CODE,
        INGESTION_TS,
        SOURCE_FILE_NAME
    FROM STREAM_ORDER_DETAILS;

    ------------------------------------------------------------------
    -- BAD RECORDS
    ------------------------------------------------------------------

    INSERT INTO QUARANTINE.BAD_ORDER_RECORDS
    (
        ORDER_ID,
        CUSTOMER_ID,
        RESTAURANT_ID,
        AGENT_ID,
        ORDER_PLACED_AT,
        ORDER_ACCEPTED_AT,
        ORDER_DELIVERED_AT,
        ORDER_STATUS,
        TOTAL_AMOUNT,
        DISCOUNT_AMOUNT,
        DELIVERY_FEE,
        TAX_AMOUNT,
        FINAL_AMOUNT,
        DELIVERY_DISTANCE_KM,
        ESTIMATED_DELIVERY_TIME,
        ACTUAL_DELIVERY_TIME,
        DELIVERY_CITY,
        DELIVERY_PINCODE,
        ORDER_SOURCE,
        PROMO_CODE,
        INGESTION_TS,
        SOURCE_FILE_NAME
    )
    SELECT DISTINCT
        o.ORDER_ID,
        o.CUSTOMER_ID,
        o.RESTAURANT_ID,
        o.AGENT_ID,
        o.ORDER_PLACED_AT,
        o.ORDER_ACCEPTED_AT,
        o.ORDER_DELIVERED_AT,
        o.ORDER_STATUS,
        o.TOTAL_AMOUNT,
        o.DISCOUNT_AMOUNT,
        o.DELIVERY_FEE,
        o.TAX_AMOUNT,
        o.FINAL_AMOUNT,
        o.DELIVERY_DISTANCE_KM,
        o.ESTIMATED_DELIVERY_TIME,
        o.ACTUAL_DELIVERY_TIME,
        o.DELIVERY_CITY,
        o.DELIVERY_PINCODE,
        o.ORDER_SOURCE,
        o.PROMO_CODE,
        o.INGESTION_TS,
        o.SOURCE_FILE_NAME

    FROM TMP_ORDER_STREAM o

    LEFT JOIN SILVER.CUSTOMER_CLEAN c
        ON o.CUSTOMER_ID = c.CUSTOMER_ID

    LEFT JOIN SILVER.RESTAURANT_CLEAN r
        ON o.RESTAURANT_ID = r.RESTAURANT_ID

    LEFT JOIN SILVER.AGENT_CLEAN a
        ON o.AGENT_ID = a.AGENT_ID

    LEFT JOIN (
        SELECT DISTINCT PROMO_CODE
        FROM SILVER.PROMOTION_CLEAN
    ) p
        ON o.PROMO_CODE = p.PROMO_CODE

    WHERE

        o.ORDER_ID IS NULL
        OR o.CUSTOMER_ID IS NULL
        OR o.RESTAURANT_ID IS NULL
        OR o.ORDER_STATUS IS NULL
        OR o.ORDER_PLACED_AT IS NULL

        OR o.TOTAL_AMOUNT <= 0
        OR o.DISCOUNT_AMOUNT < 0
        OR o.TAX_AMOUNT < 0
        OR o.FINAL_AMOUNT < 0
        OR o.DELIVERY_FEE < 0
        OR o.DELIVERY_DISTANCE_KM <= 0

        OR (
            o.ORDER_ACCEPTED_AT IS NOT NULL
            AND o.ORDER_ACCEPTED_AT < o.ORDER_PLACED_AT
        )

        OR (
            o.ORDER_DELIVERED_AT IS NOT NULL
            AND o.ORDER_ACCEPTED_AT IS NOT NULL
            AND o.ORDER_DELIVERED_AT < o.ORDER_ACCEPTED_AT
        )

        OR UPPER(TRIM(o.ORDER_STATUS))
            NOT IN (
                'PLACED',
                'ACCEPTED',
                'PREPARING',
                'PICKED_UP',
                'DELIVERED',
                'CANCELLED'
            )

        OR UPPER(TRIM(o.ORDER_SOURCE))
            NOT IN ('ANDROID','IOS','WEB')

        OR (
            UPPER(TRIM(o.ORDER_STATUS)) = 'DELIVERED'
            AND o.ORDER_DELIVERED_AT IS NULL
        )

        OR (
            UPPER(TRIM(o.ORDER_STATUS)) = 'PLACED'
            AND o.ORDER_ACCEPTED_AT IS NOT NULL
        )

        OR (
            UPPER(TRIM(o.ORDER_STATUS)) = 'PICKED_UP'
            AND o.AGENT_ID IS NULL
        )

        OR c.CUSTOMER_ID IS NULL
        OR r.RESTAURANT_ID IS NULL
        OR (o.AGENT_ID IS NOT NULL AND a.AGENT_ID IS NULL)
        OR (o.PROMO_CODE IS NOT NULL AND p.PROMO_CODE IS NULL);

    ------------------------------------------------------------------
    -- GOOD RECORDS
    ------------------------------------------------------------------

    INSERT INTO SILVER.ORDER_CLEAN
    (
        ORDER_ID,
        CUSTOMER_ID,
        RESTAURANT_ID,
        AGENT_ID,
        ORDER_PLACED_AT,
        ORDER_ACCEPTED_AT,
        ORDER_DELIVERED_AT,
        ORDER_STATUS,
        TOTAL_AMOUNT,
        DISCOUNT_AMOUNT,
        DELIVERY_FEE,
        TAX_AMOUNT,
        FINAL_AMOUNT,
        DELIVERY_DISTANCE_KM,
        ESTIMATED_DELIVERY_TIME,
        ACTUAL_DELIVERY_TIME,
        DELIVERY_CITY,
        DELIVERY_PINCODE,
        ORDER_SOURCE,
        PROMO_CODE,
        INGESTION_TS,
        SOURCE_FILE_NAME
    )

    SELECT DISTINCT

        TRIM(o.ORDER_ID),
        TRIM(o.CUSTOMER_ID),
        TRIM(o.RESTAURANT_ID),
        TRIM(o.AGENT_ID),

        o.ORDER_PLACED_AT,
        o.ORDER_ACCEPTED_AT,
        o.ORDER_DELIVERED_AT,

        UPPER(TRIM(o.ORDER_STATUS)),

        o.TOTAL_AMOUNT,
        o.DISCOUNT_AMOUNT,
        o.DELIVERY_FEE,
        o.TAX_AMOUNT,
        o.FINAL_AMOUNT,

        o.DELIVERY_DISTANCE_KM,
        o.ESTIMATED_DELIVERY_TIME,
        o.ACTUAL_DELIVERY_TIME,

        INITCAP(TRIM(o.DELIVERY_CITY)),

        TRIM(o.DELIVERY_PINCODE),

        CASE
            WHEN UPPER(TRIM(o.ORDER_SOURCE))
                IN ('ANDROID APP','ANDROID')
                THEN 'ANDROID'

            WHEN UPPER(TRIM(o.ORDER_SOURCE)) = 'IOS'
                THEN 'IOS'

            WHEN UPPER(TRIM(o.ORDER_SOURCE)) = 'WEB'
                THEN 'WEB'
        END,

        TRIM(o.PROMO_CODE),

        o.INGESTION_TS,
        o.SOURCE_FILE_NAME

    FROM TMP_ORDER_STREAM o

    INNER JOIN SILVER.CUSTOMER_CLEAN c
        ON o.CUSTOMER_ID = c.CUSTOMER_ID

    INNER JOIN SILVER.RESTAURANT_CLEAN r
        ON o.RESTAURANT_ID = r.RESTAURANT_ID

    LEFT JOIN SILVER.AGENT_CLEAN a
        ON o.AGENT_ID = a.AGENT_ID

    LEFT JOIN (
        SELECT DISTINCT PROMO_CODE
        FROM SILVER.PROMOTION_CLEAN
    ) p
        ON o.PROMO_CODE = p.PROMO_CODE

    WHERE

        o.ORDER_ID IS NOT NULL
        AND o.CUSTOMER_ID IS NOT NULL
        AND o.RESTAURANT_ID IS NOT NULL
        AND o.ORDER_STATUS IS NOT NULL
        AND o.ORDER_PLACED_AT IS NOT NULL

        AND o.TOTAL_AMOUNT > 0
        AND o.DISCOUNT_AMOUNT >= 0
        AND o.TAX_AMOUNT >= 0
        AND o.FINAL_AMOUNT >= 0
        AND o.DELIVERY_FEE >= 0
        AND o.DELIVERY_DISTANCE_KM > 0

        AND (
            o.ORDER_ACCEPTED_AT IS NULL
            OR o.ORDER_ACCEPTED_AT >= o.ORDER_PLACED_AT
        )

        AND (
            o.ORDER_DELIVERED_AT IS NULL
            OR o.ORDER_ACCEPTED_AT IS NULL
            OR o.ORDER_DELIVERED_AT >= o.ORDER_ACCEPTED_AT
        )

        AND UPPER(TRIM(o.ORDER_STATUS))
            IN (
                'PLACED',
                'ACCEPTED',
                'PREPARING',
                'PICKED_UP',
                'DELIVERED',
                'CANCELLED'
            )

        AND UPPER(TRIM(o.ORDER_SOURCE))
            IN ('ANDROID','ANDROID APP','IOS','WEB')

        AND NOT (
            UPPER(TRIM(o.ORDER_STATUS)) = 'DELIVERED'
            AND o.ORDER_DELIVERED_AT IS NULL
        )

        AND NOT (
            UPPER(TRIM(o.ORDER_STATUS)) = 'PLACED'
            AND o.ORDER_ACCEPTED_AT IS NOT NULL
        )

        AND NOT (
            UPPER(TRIM(o.ORDER_STATUS)) = 'PICKED_UP'
            AND o.AGENT_ID IS NULL
        )

        AND (o.AGENT_ID IS NULL OR a.AGENT_ID IS NOT NULL)
        AND (o.PROMO_CODE IS NULL OR p.PROMO_CODE IS NOT NULL)

        AND NOT EXISTS (
            SELECT 1
            FROM SILVER.ORDER_CLEAN s
            WHERE s.ORDER_ID = o.ORDER_ID
        );

    RETURN 'ORDER STREAM PROCESSED';

END;
$$;

In [ ]:
%%sql -r dataframe_10
CALL PROCESS_ORDER_STREAM();

In [ ]:
%%sql -r dataframe_12
SELECT COUNT(*)
FROM SILVER.ORDER_CLEAN
WHERE SOURCE_FILE_NAME = 'Batch_3_Order_Details.csv';

In [ ]:
%%sql -r dataframe_13
SELECT COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS
WHERE SOURCE_FILE_NAME='Batch_3_Order_Details.csv';

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE TASK TASK_ORDER_PIPELINE
WAREHOUSE = FD_WH
WHEN SYSTEM$STREAM_HAS_DATA('STREAM_ORDER_DETAILS')
AS
CALL PROCESS_ORDER_STREAM();

In [ ]:
%%sql -r dataframe_15
ALTER TASK TASK_ORDER_PIPELINE RESUME;

In [ ]:
%%sql -r dataframe_7
SHOW TASKS LIKE 'TASK_ORDER_PIPELINE';

In [ ]:
%%sql -r dataframe_11
ALTER TASK TASK_ORDER_PIPELINE SUSPEND;

In [ ]:
%%sql -r dataframe_16
SHOW WAREHOUSES;

In [ ]:
%%sql -r dataframe_17
USE WAREHOUSE FD_WH;

In [ ]:
%%sql -r dataframe_19
SHOW GRANTS ON WAREHOUSE FD_WH;

In [ ]:
%%sql -r dataframe_20
ALTER WAREHOUSE FD_WH RESUME;

In [ ]:
%%sql -r dataframe_21
SELECT CURRENT_ROLE(), CURRENT_USER();

In [ ]:
%%sql -r dataframe_22
GRANT USAGE ON WAREHOUSE FD_WH TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r dataframe_23
GRANT CREATE STREAMLIT ON SCHEMA FOOD_DELIVERY_DB.PUBLIC TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r dataframe_18
GRANT USAGE ON DATABASE FOOD_DELIVERY_DB TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r dataframe_24
GRANT USAGE ON SCHEMA FOOD_DELIVERY_DB.PUBLIC TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r dataframe_27
GRANT USAGE ON WAREHOUSE FD_WH TO ROLE PUBLIC;

In [ ]:
%%sql -r dataframe_25
SHOW WAREHOUSES LIKE 'FD_WH';


In [ ]:
%%sql -r dataframe_26
select 1

In [ ]:
%%sql -r dataframe_28
